# 03 · Windows and physics features (first half)

**Note (26 Sep 2026):** this notebook records Stage 2A–2B on the band-stop-only pipeline, loaded from
`configs/variants/r1_bandstop_only.yaml`. The primary pipeline is now r1cap10 (section 6), which adds the 10 kHz low-pass.

## Purpose
Build and verify the shared preprocessing that every model will see, on a small training subset only:
whole-recording cleaning (microphone events and a fixed band-stop for the ~22 kHz line), windowing with per-window
shaft speed, and the physics features (R1). No full caches, no predictive models and no spectrograms here; the only
model is a diagnostic RandomForest that checks whether the band-stop removed a shortcut.

## Inputs
`metadata.csv`, `data/raw_npy/`, `configs/base.yaml`, and the caches written by `scripts/06_bandstop.py`
(band-stop measurements) and `scripts/07_features.py` (subset features), each named by the sha256 of what produced it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tools, the settings and <code>metadata.csv</code>, and finds the cached results of the band-stop script. If a cache is missing it stops and says which command to run.</div>

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import freqz, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

from ml.cache import load_features, read_cache_metadata
from ml.config import config_sha256, load_config
from ml.features import feature_columns, feature_dictionary, line_overlaps, line_tolerance_hz
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, cache_file, data_dir
from ml.preprocess import apply_fir_centred, bandstop_taps, propose_stopband
from ml.splits import split_hash

cfg = load_config(DEFAULT_CONFIG)
# This notebook documents Stage 2A-2B, built on the band-stop-only pipeline (primary until 26 Sep 2026). Since then
# the primary is r1cap10 (base.yaml); the earlier pipeline is loaded explicitly from its variant file.
pre = load_config(DEFAULT_CONFIG.parent / "variants" / "r1_bandstop_only.yaml").preprocess
FS = pre.fs_hz
bs = pre.bandstop
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")

caches = {
    "measure": cache_file("06_measure", pre.mic, ext="parquet"),
    "check": cache_file("06_check", pre.mic, ext="parquet"),
    "verify": cache_file("06_verify", pre.mic, pre.bandstop, ext="parquet"),
    "shortcut": cache_file("06_shortcut", pre.mic, pre.bandstop, ext="parquet"),
}
missing = [k for k, p in caches.items() if not p.exists()]
assert not missing, f"run: python scripts/06_bandstop.py {' '.join(missing)}"
print(f"preprocess config sha256 {config_sha256(pre)[:16]}…   split hash {split_hash(meta)[:12]}…")

## 1. The band-stop for the ~22 kHz line

**Exposure note.** Stage 1c (notebook 02) already measured this line across *all* classes, including faults, as an
exploratory check. That is how its folder-linked level was found. The stop band below is nevertheless **set from
`split_closed` train normal recordings only**; fault recordings are used afterwards only to *check* it, never to widen
it. Normal is never withheld, so this setting is safe for every LOCO fold and for the healthy-only track.

### Setting it
Per train normal recording and channel 1–7: the main line (strongest bin in 21.8–22.3 kHz, 0.2 Hz bins, interpolated)
and its ±60/±120 Hz sidebands, plus the line's drift across 1 s windows. The rule: cover the observed extent of the
lines, add a stated **25 Hz margin** each side (about the spread of the line across recordings), round outward to 5 Hz.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tone measurements of the 34 train normal recordings, summarises where the line and its sidebands sit, applies the stop-band rule, and checks the result equals the band stored in the settings.</div>

In [ ]:
tone_n = pd.read_parquet(caches["measure"])
present = tone_n[tone_n.snr_db > 20]
print(f"{tone_n.rec_id.nunique()} train normal recordings, {len(present)} of {len(tone_n)} recording × channel pairs "
      f"with the line > 20 dB above the local floor")
summary = pd.DataFrame({
    "frequency min (Hz)": [present.f0_hz.min()] + [present[f"f{o}_hz"].min() for o in ("-120", "-60", "+60", "+120")],
    "frequency max (Hz)": [present.f0_hz.max()] + [present[f"f{o}_hz"].max() for o in ("-120", "-60", "+60", "+120")],
}, index=["line", "−120 Hz", "−60 Hz", "+60 Hz", "+120 Hz"]).round(1)
print(f"drift of the line within a recording (1 s windows): median {present.drift_hz.median():.2f} Hz, "
      f"max {present.drift_hz.max():.2f} Hz")
band = propose_stopband(tone_n, present_db=20, margin_hz=25, round_hz=5)
print(band)
assert (band["stop_lo_hz"], band["stop_hi_hz"]) == (bs.stop_lo_hz, bs.stop_hi_hz), "configs/base.yaml bandstop is out of date"
summary

The filter: a symmetric, odd-length **linear-phase FIR** (`scipy.signal.firwin`, Kaiser window), applied by centred
convolution so its group delay is removed (zero-phase alignment). Design targets: 80 dB across the stop band and
200 Hz transitions each side, which `kaiserord` turns into 1,257 taps (12.6 ms half-length) and Kaiser β = 7.857.
Edges, tap count and window are stored in `bandstop` in the config.

### Checking it on other train and val recordings (no retuning)
The same measurement on every other train and val recording, including fault recordings. Anything outside the stop
band is reported, not used to widen it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the same tone measurements for all other train and val recordings and counts, for the line and each sideband, how many recording × channel pairs fall outside the stop band, and in which classes.</div>

In [ ]:
tone_c = pd.read_parquet(caches["check"])
pc = tone_c[tone_c.snr_db > 20]
rows = []
for name, fcol, scol, thr in [("line", "f0_hz", "snr_db", 20)] + [
        (f"{o} Hz", f"f{o}_hz", f"snr{o}_db", 10) for o in ("-120", "-60", "+60", "+120")]:
    s = pc[pc[scol] > thr]
    out = s[(s[fcol] < bs.stop_lo_hz) | (s[fcol] > bs.stop_hi_hz)]
    rows.append({"line": name, "present": len(s), "outside the stop band": len(out), "files": out.rec_id.nunique(),
                 "range (Hz)": f"{out[fcol].min():.0f}–{out[fcol].max():.0f}" if len(out) else "",
                 "classes (files)": out.groupby("class10").rec_id.nunique().to_dict() if len(out) else ""})
print(f"{tone_c.rec_id.nunique()} other train/val recordings; line frequency {pc.f0_hz.min():.1f}–{pc.f0_hz.max():.1f} Hz")
pd.DataFrame(rows)

**The line itself always falls inside the stop band**, but in some sessions it sits higher than in any train normal
recording (up to 22,156 Hz), so its **+120 Hz sideband lands in the upper transition band in 46 recordings** (up to
22,276 Hz; 26 of them horizontal misalignment) and the +60 Hz sideband in 5 (all horizontal misalignment). There the
attenuation is partial. Reported, not fixed: widening the band from fault recordings is not allowed, and any later
data-driven change must respect the relevant fold.

### (a) The designed frequency response

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes the filter's frequency response on a fine grid and reports the smallest attenuation inside the stop band, the largest ripple outside the transitions, and where the response crosses −0.01, −3 and −80 dB.</div>

In [ ]:
taps = bandstop_taps(pre)
w, h = freqz(taps, worN=2**20, fs=FS)
H = 20 * np.log10(np.abs(h) + 1e-15)
in_stop = (w >= bs.stop_lo_hz) & (w <= bs.stop_hi_hz)
passband = (w <= bs.stop_lo_hz - bs.transition_hz) | (w >= bs.stop_hi_hz + bs.transition_hz)


def crossing(level: float, lower: bool) -> float:
    side = (w > 21_000) & (w < bs.stop_lo_hz + 1) if lower else (w > bs.stop_hi_hz - 1) & (w < 23_000)
    ww, hh = w[side], H[side]
    return float(ww[np.argmax(hh < level)] if lower else ww[np.argmax(hh > level)])


resp = pd.Series({
    "taps": taps.size, "symmetric": bool(np.allclose(taps, taps[::-1])),
    "min stop-band attenuation (dB)": -H[in_stop].max(),
    "max pass-band ripple (dB)": np.abs(H[passband]).max(),
    **{f"lower edge {lvl} dB (Hz)": crossing(lvl, True) for lvl in (-0.01, -3, -80)},
    **{f"upper edge {lvl} dB (Hz)": crossing(lvl, False) for lvl in (-80, -3, -0.01)},
})
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
axes[0].plot(w / 1e3, H, lw=0.8)
axes[0].set(xlabel="frequency (kHz)", ylabel="gain (dB)", ylim=(-120, 5), title="Band-stop response, 0–25 kHz")
axes[1].plot(w, H, lw=0.8)
axes[1].axvspan(bs.stop_lo_hz, bs.stop_hi_hz, color="red", alpha=0.1, label="stop band")
axes[1].set(xlim=(21_500, 22_600), ylim=(-120, 5), xlabel="frequency (Hz)", title="zoom")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG / "03_bandstop_response.png")
resp.round(4)

### (b) Synthetic signals: a tone at the stop-band frequency plus broadband noise

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds 5 s test signals of white noise plus a tone at the typical line frequency, at the upper end seen in fault sessions, and at both stop-band edges; filters them; and reports how much each tone drops and how much the noise spectrum changes outside the stop band and transitions.</div>

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(5 * FS) / FS
half = (bs.numtaps - 1) // 2
rows = []
for f_tone in (22_028.0, 22_156.0, bs.stop_lo_hz, bs.stop_hi_hz):
    noise = 0.05 * rng.normal(size=t.size)
    tone_sig = 0.5 * np.sin(2 * np.pi * f_tone * t)
    y = apply_fir_centred(noise + tone_sig, taps)
    core = slice(half, t.size - half)
    probe = np.exp(-2j * np.pi * f_tone * t[core])
    a_in = 2 * np.abs(np.mean((noise + tone_sig)[core] * probe))
    a_out = 2 * np.abs(np.mean(y[core] * probe))
    fw, p_in = welch(noise[core], fs=FS, nperseg=8192)
    _, p_out = welch(apply_fir_centred(noise, taps)[core], fs=FS, nperseg=8192)
    keep = ((fw < bs.stop_lo_hz - bs.transition_hz) | (fw > bs.stop_hi_hz + bs.transition_hz)) & (fw > 20)
    rows.append({"tone (Hz)": f_tone, "tone attenuation (dB)": 20 * np.log10(a_in / a_out),
                 "noise change outside (max |dB|)": np.abs(10 * np.log10(p_out[keep] / p_in[keep])).max()})
pd.DataFrame(rows).round(3)

At the stop-band edges the tone drops by 80.9–81.0 dB, matching the 80 dB design; inside the band it drops further (82.7 dB at 22,028 Hz, 98.6 dB at 22,156 Hz). The noise spectrum outside the stop band and its transitions changes by at most 0.001 dB.

### (c) Real train recordings
Three train recordings per class (30), from the scan cache: the line's attenuation per recording (where it is present),
the largest change outside the stop band and its transitions (100 Hz bands), and the **measured** recording-edge
effect. The edge effect is measured by filtering a 2 s crop as if it were a recording and comparing it with the same
2 s cut from the filtered full recording (which had real signal on both sides).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the per-recording verification results and summarises them: minimum, median and maximum over the 30 train recordings, for the line attenuation, the change outside the stop band, and how many milliseconds from each recording edge the filter's output stays more than −40 dB and −60 dB away from the true result.</div>

In [ ]:
ver = pd.read_parquet(caches["verify"])
print(f"{len(ver)} train recordings; filter half-length {(bs.numtaps - 1) / 2 / FS * 1e3:.1f} ms")
ver.drop(columns=["rec_id", "class10"]).describe().T[["min", "50%", "max"]].round(4)

The edge effect is shorter than the filter's half-length: the output is within −40 dB of the true result after
4–7 ms and within −60 dB after 8–11 ms, at both ends. The first and last 1 s windows therefore contain at most ~11 ms of
slightly disturbed signal. Because the band-stop runs on the whole recording (rule 11), only those two recording edges
are affected, never the window boundaries inside a recording.

### Diagnostic shortcut test
Is the stop band still class-informative after filtering? Per-channel energy in the stop band (log) → class10,
RandomForest, 5-fold `cv_fold` CV on train + val, against a label-permutation baseline (10 permutations). As a further
diagnostic, the same test on two bands next to the stop band (21.3–21.6 and 22.45–22.75 kHz) that the filter does not
touch. The result is a finding to report, not a reason to retune the filter.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the per-recording stop-band energies from the scan cache, trains a random forest with 5-fold cross-validation on the energies before filtering, after filtering, and in the untouched neighbouring bands, and compares each macro-F1 with the score when the class labels are randomly shuffled.</div>

In [ ]:
sc = pd.read_parquet(caches["shortcut"])
chans = list(pre.channels[1:])
y = sc.class10.to_numpy()


def oof_f1(X: np.ndarray, labels: np.ndarray) -> float:
    pred = np.empty(len(sc), dtype=object)
    for f in range(5):
        tr, te = (sc.cv_fold != f).to_numpy(), (sc.cv_fold == f).to_numpy()
        pred[te] = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X[tr], labels[tr]).predict(X[te])
    return f1_score(labels, pred, average="macro")


feats = {k: np.log10(sc[[f"{k}_{c}" for c in chans]].to_numpy() + 1e-20)
         for k in ("before", "after", "before_adjacent")}
rng = np.random.default_rng(0)
perm = [oof_f1(feats["after"], rng.permutation(y)) for _ in range(10)]
shortcut_bs = pd.Series({
    "stop-band energy before filtering": oof_f1(feats["before"], y),
    "stop-band energy after filtering": oof_f1(feats["after"], y),
    "neighbouring bands (untouched)": oof_f1(feats["before_adjacent"], y),
    "permutation baseline, mean": np.mean(perm),
    "permutation baseline, max of 10": np.max(perm),
})
print(f"{len(sc)} train + val recordings, 10 classes (chance ≈ 0.10)")
shortcut_bs.round(3)

**Finding.** The stop band's energy predicts the class almost perfectly before filtering (macro-F1 0.98). After filtering it still reaches 0.906, far above the permutation baseline (0.10), and the untouched neighbouring bands alone reach 0.935. So stop-band energy remains class-informative after filtering, but **the cause is not established**: residual tone, background characteristics and other acquisition artefacts may all contribute. The band-stop removes the line itself as designed; what remains is a shortcut risk for any representation that reaches these frequencies (Stage 2C).

## 2. Windows and per-window speed
Cleaning runs on the whole recording (rule 11); only then is the recording cut into 9 windows of 1 s with a 0.5 s hop,
with DC removed per window on every channel except the tachometer. Each window gets its own shaft speed from the clean
tachometer intervals inside it (`window_speeds_detail`, on the **untouched** tachometer), falling back to the recording
speed when too few clean intervals fall inside. The fallback flag and speed variability are metadata, never model
inputs.

## 3. The physics features (R1)
`ml.features.recording_features(x_raw, fr_hz, cfg)` is the only entry point, used by batch and serving alike:
clean the recording → window speeds → whole-recording envelopes (1–5 kHz band-pass + Hilbert) on under_radial and
over_radial → aligned signal and envelope windows → `physics_features`. Feature columns start with `f_`; model code
selects them by that prefix only.

**Line search tolerance.** 1 s windows give 1 Hz bins: 0.083 orders at 12 Hz, 0.017 orders at 60 Hz. Each spectral line
is searched within ±max(1.5 bins, 0.5 % of the line frequency): the bin term covers the Hann main lobe at low
frequency, the relative term the growing effect of speed error at high orders. 4× zero-padding interpolates the
spectrum between bins; it adds no resolution.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the feature dictionary (one row per feature), counts features by unit and by channel group, and shows the search tolerance at the slowest and fastest speeds for a few lines.</div>

In [ ]:
fd = feature_dictionary(pre)
print(f"{len(fd)} features, version {pre.features.version}; dictionary: results/feature_dictionary_r1-1.0.csv")
group = np.select([fd.name.str.startswith("f_env_"), fd.name.str.startswith("f_mic_")], ["envelope", "microphone"],
                  "accelerometer")
display(pd.crosstab(group, fd.unit))
tol = pd.DataFrame({f"{sp:g} Hz": {f"{o:g}×": line_tolerance_hz(o * sp, pre) for o in (0.5, 1, 3, 10)}
                    for sp in (11.8, 30.0, 60.7)})
tol.index.name = "line"
tol.round(2).rename_axis(columns="± tolerance (Hz) at shaft speed")

**Where neighbouring lines' search windows overlap**, at the slowest measured speed and at 60 Hz:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Lists every pair of spectral lines whose search windows overlap at 11.8 Hz and at 60 Hz. An overlapping pair can pick up the same peak, so its two features are not independent.</div>

In [ ]:
pd.concat({f"{sp:g} Hz": line_overlaps(sp, pre) for sp in (11.8, 60.0)}, names=["speed"]).round(3)

No accelerometer orders overlap. Among the envelope lines, **BPFO×2 / BPFI+1×** and **BPFO×3 / BPFI×2−1×** overlap
at every speed: they lie 0.006 and 0.010 orders apart, far closer than any 1 s window can resolve. On the subset they
are **highly correlated, not exact duplicates** (identical in 850–1,047 of 1,278 windows, r = 0.984–0.997). They are
kept for the first baseline; any later feature selection uses training data only. **2×BSF+FTF / BPFI−1×** (0.115 orders apart) overlap only at low speed (below about 14 Hz).

## 4. Subset run
`scripts/07_features.py subset --fraction 0.1`: ceil(10 %) of the `split_closed` **train** recordings of each class
(fixed seed), through the batch path, into a cache named by `ml.cache.feature_cache_key`. Reading it back recomputes
the key from the current config, feature definitions, source checksums and supplied speeds, and fails loudly if
anything changed. The cache holds no split assignments; they are joined from `metadata.csv` when it is loaded.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the subset feature cache through the identity check, joins the current labels and split assignments from <code>metadata.csv</code>, prints its provenance (git sha, runtime), the runtime per recording and the projected time for all 1,951 recordings, and checks for missing or infinite values and for exactly 9 windows per recording.</div>

In [ ]:
path = max((data_dir() / "cache").glob("r1_subset_*.parquet"), key=lambda p: p.stat().st_mtime)
feat = load_features(path, meta, pre)  # validates the cache key; joins the current labels and splits
prov = read_cache_metadata(path)
fcols = feature_columns(feat)
per_rec = feat.groupby("rec_id").compute_s.first()
workers = int(prov["workers"])
print(f"{path.name}: {feat.rec_id.nunique()} recordings, {len(feat)} windows, {len(fcols)} features")
print(f"git {prov['git_sha'][:12]}, feature version {prov['feature_version']}; splits joined from metadata.csv "
      f"(split hash {split_hash(meta)[:12]}…)")
print(f"runtime: {per_rec.mean():.2f} s per recording on one core (max {per_rec.max():.2f}); {prov['wall_s']} s wall "
      f"with {workers} workers → all 1,951 recordings ≈ {1951 * per_rec.mean() / workers / 60:.1f} min wall")
checks = pd.Series({
    "NaN or inf values": int((~np.isfinite(feat[fcols].to_numpy())).sum()),
    "recordings without exactly 9 windows": int((feat.groupby("rec_id").size() != 9).sum()),
    "windows on fallback speed": int(feat.speed_fallback.sum()),
    "recordings with any fallback window": int(feat.groupby("rec_id").speed_fallback.any().sum()),
    "non-train recordings": int((feat.split_closed != "train").sum()),
})
display(feat.groupby("class10").rec_id.nunique().rename("recordings").to_frame().T)
checks

Feature ranges (over all subset windows), a few per group:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the minimum, median and maximum of a selection of features across all subset windows, to spot values that are negative, zero or implausibly large.</div>

In [ ]:
pick = ["f_under_radial_rms", "f_under_radial_kurtosis", "f_under_radial_o1", "f_over_tangential_o2",
        "f_under_radial_ratio_2x_1x", "f_under_ratio_axial_radial_1x", "f_env_under_radial_bpfo1",
        "f_env_over_radial_ftf1", "f_mic_rms", "f_mic_centroid_hz", "f_mic_rolloff_hz", "f_mic_logband_2000_4000"]
rng_tab = feat[fcols].describe(percentiles=[0.5]).T[["min", "50%", "max"]]
print(f"features with a negative minimum: {int((rng_tab['min'] < 0).sum())} "
      f"(kurtosis and log band energies can be negative); exactly zero anywhere: {int((feat[fcols] == 0).any().sum())}")
rng_tab.loc[pick].round(4)

## 5. Sanity plots
Per recording, the median over its 9 windows.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws three checks against known physics: the 1× amplitude on two underhang channels for normal and imbalance recordings by added mass, against shaft speed; the envelope BPFO line for outer-race and normal recordings on both radial channels; and the 2× amplitude and 2×/1× ratio for misalignment against normal.</div>

In [ ]:
rec = feat.groupby(["rec_id", "class10", "severity", "fr_measured_hz"])[fcols].median().reset_index()
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
imb = rec[rec.class10.isin(["normal", "imbalance"])]
sc_ = axes[0].scatter(imb.fr_measured_hz, imb.f_under_radial_o1, c=imb.severity, cmap="viridis", s=30)
axes[0].set(yscale="log", xlabel="shaft speed (Hz)", ylabel="under_radial 1× amplitude (V)",
            title="Imbalance: 1× by mass (0 g = normal)")
fig.colorbar(sc_, ax=axes[0], label="added mass (g)")
groups = ["normal", "underhang_outer", "overhang_outer"]
for k, ch in enumerate(["under_radial", "over_radial"]):
    vals = [rec.loc[rec.class10 == g, f"f_env_{ch}_bpfo1"] for g in groups]
    axes[1].boxplot(vals, positions=np.arange(3) + k * 0.35 - 0.17, widths=0.3, patch_artist=True,
                    boxprops={"facecolor": f"C{k}", "alpha": 0.5}, showfliers=False)
axes[1].set_xticks(range(3), [f"{g}\n(n={int((rec.class10 == g).sum())})" for g in groups], fontsize=8)
axes[1].set(yscale="log", ylabel="envelope BPFO / envelope median", title="BPFO line, 1–5 kHz band (C0 under, C1 over)")
mis = ["normal", "horizontal_misalignment", "vertical_misalignment"]
axes[2].boxplot([rec.loc[rec.class10 == g, "f_over_tangential_o2"] for g in mis], showfliers=False)
axes[2].set_xticks(range(1, 4), [f"{g}\n(n={int((rec.class10 == g).sum())})" for g in mis], fontsize=8)
axes[2].set(yscale="log", ylabel="over_tangential 2× amplitude (V)", title="Misalignment: 2×")
fig.tight_layout()
fig.savefig(FIG / "03_sanity.png")
summ = pd.DataFrame({
    "under_radial 1× (V)": rec.groupby("class10").f_under_radial_o1.median(),
    "env BPFO under_radial": rec.groupby("class10").f_env_under_radial_bpfo1.median(),
    "env BPFO over_radial": rec.groupby("class10").f_env_over_radial_bpfo1.median(),
    "over_tangential 2× (V)": rec.groupby("class10").f_over_tangential_o2.median(),
    "over_tangential 2×/1×": rec.groupby("class10").f_over_tangential_ratio_2x_1x.median(),
}).reindex(CLASS10)
display(rec[rec.class10.isin(["normal", "imbalance"])].groupby("severity").f_under_radial_o1.agg(["size", "median"]).round(3).T)
summ.round(3)

- **Imbalance:** the 1× amplitude rises with added mass, from 0.16 V (normal) to 0.37 V (6 g) and 1.9 V (20 g), but not
  monotonically above 20 g in this subset (0.78 V at 35 g): with about 3 recordings per mass, the spread of speeds
  (imbalance force grows with speed squared) dominates. Against speed, heavier masses mostly sit above lighter ones at similar speeds, with one clear exception:
  `imbalance__30g__14.9504` has only 0.020 V at 1× on under_radial, while its over_tangential 1× (0.29 V) looks
  normal for its mass; it has no quality flags and a steady speed. Worth watching, not explained here.
- **Outer race, BPFO line in the 1–5 kHz band:** weak. On over_radial the median rises from 2.4 (normal) to 3.8 (underhang
  outer) and 3.1 (overhang outer); on under_radial, normal is already high (7.5), because BPFO ≈ 3× shaft speed and the
  3rd shaft order is present in healthy envelopes too. This matches Stage 1c: the in-range band carries less outer-race
  evidence than the out-of-range comparison bands. Only 4 normal recordings are in the subset.
- **Misalignment:** the 2× amplitude on over_tangential doubles for horizontal (0.048 V against 0.023 V) and more than
  quadruples for vertical misalignment (0.106 V), as in Stage 1c.

### Suspect channel: `imbalance__30g__14.9504`, under_radial
Its 1× amplitude on under_radial is far below the other imbalance recordings. Is the channel dead or disconnected?
Compared with the nearest recordings of the same 30 g folder that are **not** in the test split (13.1 Hz, val and
16.6 Hz, val; the one at 14.3 Hz is a test recording and is not opened).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the suspect recording and its two neighbours from the same folder, plots 0.5 s of their raw underhang radial signal and the spectrum around the first three shaft orders, and tabulates the channel's standard deviation, number of distinct values and 1× amplitude for each.</div>

In [ ]:
from ml.features import recording_features

suspect = ["imbalance__30g__13.1072", "imbalance__30g__14.9504", "imbalance__30g__16.5888"]
m_idx = meta.set_index("rec_id")
ur = pre.channels.index("under_radial")
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
rows = []
for rid in suspect:
    x = np.load(data_dir() / "raw_npy" / f"{rid}.npy")
    fr = m_idx.loc[rid, "fr_measured_hz"]
    n = FS // 2
    axes[0].plot(np.arange(n) / FS, x[:n, ur], lw=0.5, label=f"{rid} ({m_idx.loc[rid, 'split_closed']})")
    sig = x[:, ur] - x[:, ur].mean()
    w = np.hanning(sig.size)
    amp = 2 * np.abs(np.fft.rfft(sig * w)) / w.sum()
    fq = np.fft.rfftfreq(sig.size, 1 / FS)
    keep = fq < 3.5 * fr
    axes[1].semilogy(fq[keep] / fr, amp[keep], lw=0.7, label=rid)
    f1 = recording_features(x, fr, pre)
    rows.append({"rec_id": rid, "split": m_idx.loc[rid, "split_closed"], "speed (Hz)": fr,
                 "std (V)": x[:, ur].std(), "distinct values": len(np.unique(x[:, ur])),
                 "1× median over windows (V)": f1.f_under_radial_o1.median(),
                 "1× min–max over windows (V)": f"{f1.f_under_radial_o1.min():.3f}–{f1.f_under_radial_o1.max():.3f}"})
axes[0].set(xlabel="time (s)", ylabel="under_radial (V)", title="Raw under_radial, first 0.5 s")
axes[0].legend(fontsize=7)
axes[1].set(xlabel="order (× shaft speed)", ylabel="amplitude (V)", title="Spectrum up to 3.5 orders")
fig.tight_layout()
fig.savefig(FIG / "03_suspect_channel.png")
pd.DataFrame(rows).round(3)

**Verdict: not dead or disconnected.** The channel's level (std 0.326 V against 0.327–0.329 V for the neighbours),
its resolution (about 105,000 distinct values) and its waveform look like the neighbours'. Its low 1× (0.020 V, stable
across all 9 windows) matches the 13.1 Hz neighbour exactly (0.020 V); the 16.6 Hz neighbour has 0.057 V. At these low
speeds the 1× response of this channel in this folder is simply small (imbalance force grows with speed squared). No
quality flag is added; the file is kept.

## 6. Full feature tables and the sensitivity variant

**PRIMARY** (`data/features/r1_<key>.parquet`): the pipeline as decided, for every recording. Feature generation is
deterministic, so test recordings are included; their labels are not used anywhere here.

**SENSITIVITY VARIANT r1cap10** (`data/features/r1cap10_<key>.parquet`), declared in `configs/variants/r1cap10.yaml`
before any results: the same pipeline plus a fixed low-pass on channels 1–7 inside `clean_recording`, so it acts before
every feature, including microphone RMS and all accelerometer statistics; microphone bands stop at 10 kHz (142
features). Design: linear-phase Kaiser FIR (`firwin`), pass band to 9.8 kHz, stop band from 10.2 kHz, required ≥ 80 dB and
≤ 0.01 dB ripple. A design target of 80 dB measured only 79.73 dB, so the target is 81 dB: `kaiserord` gives 639 taps
(6.4 ms half-length, β = 7.9675), applied by centred convolution.

**Interpretation rule.** This is a sensitivity experiment. A performance drop would show that the models depend on the
removed information (above 10 kHz); it would not prove that information was an artefact.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the variant's settings, computes its low-pass filter's frequency response and reports the largest deviation in the pass band (up to 9.8 kHz), the smallest attenuation in the stop band (from 10.2 kHz), and the tap count.</div>

In [ ]:
from ml.config import load_config as _load
from ml.paths import CONFIGS_DIR
from ml.preprocess import lowpass_taps

cap = _load(CONFIGS_DIR / "variants" / "r1cap10.yaml").preprocess
lp_taps = lowpass_taps(cap)
wl, hl = freqz(lp_taps, worN=2**20, fs=FS)
Hl = 20 * np.log10(np.abs(hl) + 1e-15)
pd.Series({"taps": lp_taps.size, "symmetric": bool(np.allclose(lp_taps, lp_taps[::-1])),
           "max |pass-band deviation| ≤ 9.8 kHz (dB)": np.abs(Hl[wl <= cap.lowpass.pass_hz]).max(),
           "min stop-band attenuation ≥ 10.2 kHz (dB)": -Hl[wl >= cap.lowpass.stop_hz].max(),
           "−3 dB point (Hz)": wl[np.argmax(Hl < -3)]}).round(5)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Synthetic test: white noise plus a 5 kHz tone (should pass) and a 12 kHz tone (should be removed), filtered with the low-pass; reports each tone's change and whether the 5 kHz tone keeps its timing.</div>

In [ ]:
rng = np.random.default_rng(1)
t = np.arange(5 * FS) / FS
half_l = (lp_taps.size - 1) // 2
core = slice(half_l, t.size - half_l)
sig = 0.05 * rng.normal(size=t.size) + 0.3 * np.sin(2 * np.pi * 5_000 * t) + 0.3 * np.sin(2 * np.pi * 12_000 * t)
out = apply_fir_centred(sig, lp_taps)
tone = lambda v, f: 2 * np.abs(np.mean(v[core] * np.exp(-2j * np.pi * f * t[core])))
pd.Series({"5 kHz tone change (dB)": 20 * np.log10(tone(out, 5_000) / tone(sig, 5_000)),
           "12 kHz tone attenuation (dB)": 20 * np.log10(tone(sig, 12_000) / tone(out, 12_000)),
           "5 kHz phase shift (degrees)": np.degrees(np.angle(np.mean(out[core] * np.exp(-2j * np.pi * 5_000 * t[core]))
                                                           / np.mean(sig[core] * np.exp(-2j * np.pi * 5_000 * t[core]))))}).round(4)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the measured edge effect of the low-pass on one train recording per class (from <code>scripts/07_features.py lowpass-edge</code>) and summarises how many milliseconds from each recording end the output stays more than −40 dB and −60 dB away from the true result.</div>

In [ ]:
edge = pd.read_parquet(cache_file("07_lowpass_edge", cap.lowpass, ext="parquet"))
print(f"{len(edge)} train recordings (one per class); low-pass half-length {(cap.lowpass.numtaps - 1) / 2 / FS * 1e3:.1f} ms")
edge.filter(like="edge").describe().loc[["min", "50%", "max"]]

### Checks on both full tables

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads both full feature tables through the identity check (which also joins the current labels and splits) and checks each for missing or infinite values, exactly 9 windows per recording, and counts the windows that used the fallback speed, by class.</div>

In [ ]:
from ml.cache import load_features as _lf

tables = {}
for name, conf in (("r1", pre), ("r1cap10", cap)):
    p = max((data_dir() / "features").glob(f"{name}_*.parquet"), key=lambda q: q.stat().st_mtime)
    tables[name] = (p, _lf(p, meta, conf))
rows = {}
for name, (p, df) in tables.items():
    fc = feature_columns(df)
    rows[name] = {"file": p.name, "recordings": df.rec_id.nunique(), "windows": len(df), "features": len(fc),
                  "NaN or inf": int((~np.isfinite(df[fc].to_numpy())).sum()),
                  "recordings without 9 windows": int((df.groupby("rec_id").size() != 9).sum()),
                  "fallback windows": int(df.speed_fallback.sum())}
display(pd.DataFrame(rows).T)
fb = pd.DataFrame({n: df.groupby("class10").speed_fallback.sum() for n, (_, df) in tables.items()}).reindex(CLASS10)
fb.assign(recordings=tables["r1"][1].groupby("class10").rec_id.nunique())

Both tables are complete: 1,951 recordings × 9 windows, no missing or infinite values, 144 (primary) and 142
(variant) features. The 846 fallback-speed windows are the 94 severe tacho-glitch recordings (9 windows each), mostly
underhang ball (63 recordings).

## Findings (first half)
- **Band-stop set from 34 train normal recordings only:** 21,870–22,195 Hz (line and ±60/±120 Hz sidebands 21,899–22,166 Hz, + 25 Hz margin); Kaiser FIR, 1,257 taps, ≥ 79.9 dB in the stop band, 0.001 dB ripple. On other train/val recordings the line always falls inside, but the +120 Hz sideband lands in the transition band in 46 recordings (up to 22,276 Hz) and +60 Hz in 5; not widened.
- **Verification:** real train recordings lose ≥ 82 dB of the line, change by ≤ 0.0008 dB outside the stop band, and show a measured edge effect of 4–7 ms (−40 dB) and 8–11 ms (−60 dB) at each recording end. Stop-band energy still predicts class after filtering (macro-F1 0.906 vs a 0.10 permutation baseline), and so do the untouched neighbouring bands (0.935); the cause is not established (residual tone, background characteristics or other acquisition artefacts may contribute).
- **144 features** per window (98 accelerometer, 34 envelope, 12 microphone), all finite on the subset; lines searched within ±max(1.5 bins, 0.5 %); BPFO/BPFI pairs overlap at every speed (highly correlated, r = 0.984–0.997, not exact duplicates) and one BSF/BPFI pair below ~14 Hz.
- **Runtime:** about 0.3 s per recording on one core; the 142-recording subset took 4 s with 12 workers, so all 1,951 recordings would take under a minute. Batch and serving paths give bit-identical features.
- **Sanity:** 1× grows with imbalance mass (0.16 → 1.9 V), 2× grows with misalignment (up to 4.6×), but the 1–5 kHz BPFO line separates outer race from normal only weakly (median 2.4 → 3.8 on over_radial).

## Outputs written
- `results/figures/03_bandstop_response.png`, `results/figures/03_sanity.png`
- `results/feature_dictionary_r1-1.0.csv` and, for the primary r1cap10, `results/feature_dictionary.csv` (by `scripts/07_features.py dictionary`)
- Subset cache `data/cache/r1_subset_shared_<key>.parquet` (by `scripts/07_features.py subset`), not committed